[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/11_Advanced_Topics_and_Projects/04_End_to_End_Project/End_to_End_Project_Deep_Dive.ipynb)

# 11.4 End-to-End ML Pipeline with ONNX — Deep Dive

## Table of Contents
1. [Pipeline Formalization](#section-1)
2. [Training Phase: Loss, Optimization, Convergence](#section-2)
3. [ONNX Export: Graph Construction and Operator Mapping](#section-3)
4. [Numerical Parity Verification](#section-4)
5. [Graph Optimization Pipeline](#section-5)
6. [Quantization Pipeline: Calibration and Error Analysis](#section-6)
7. [Deployment Architecture](#section-7)
8. [Benchmarking Framework](#section-8)
9. [Monitoring and Observability](#section-9)
10. [Model Versioning and Registry Patterns](#section-10)
11. [CI/CD for ML Pipelines](#section-11)
12. [Full Working Example: CIFAR-10 CNN](#section-12)
13. [Summary](#section-13)

---

This notebook formalizes the complete ML lifecycle as a mathematical pipeline:

$$\text{Train} \xrightarrow{\text{export}} \text{ONNX} \xrightarrow{\text{optimize}} G' \xrightarrow{\text{quantize}} G_q \xrightarrow{\text{deploy}} \text{API} \xrightarrow{\text{monitor}} \text{SLOs}$$

Each stage has formal correctness criteria, and we prove or verify them throughout.

<a id='section-1'></a>
## Section 1: Pipeline Formalization

### The ML Pipeline as a Formal Composition

An end-to-end ML pipeline is a composition of transformations, each with defined inputs, outputs, and correctness invariants.

**Definition.** An ML pipeline $\Pi$ is a tuple:

$$\Pi = (\mathcal{T}, \mathcal{E}, \mathcal{O}, \mathcal{Q}, \mathcal{D}, \mathcal{M})$$

where:
- $\mathcal{T}: \mathcal{D}_{\text{train}} \to \theta^*$ — **Training**: dataset to optimized parameters
- $\mathcal{E}: (f_\theta, x_{\text{dummy}}) \to G$ — **Export**: model + sample to ONNX graph
- $\mathcal{O}: G \to G'$ — **Optimization**: graph rewriting preserving semantics
- $\mathcal{Q}: G' \to G_q$ — **Quantization**: precision reduction with calibration
- $\mathcal{D}: G_q \to \text{API}$ — **Deployment**: graph to serving endpoint
- $\mathcal{M}: \text{API} \to \text{Metrics}$ — **Monitoring**: runtime observability

### Correctness Invariants Across Stages

Each transition must satisfy a **parity constraint**:

| Transition | Invariant | Tolerance |
|:---|:---|:---:|
| Train → Export | $\|f_\theta(x) - G(x)\|_\infty \leq \epsilon_{\text{fp32}}$ | $\sim 10^{-6}$ |
| Export → Optimize | $\|G(x) - G'(x)\|_\infty \leq \epsilon_{\text{fp32}}$ | $\sim 10^{-5}$ |
| Optimize → Quantize | $\|G'(x) - G_q(x)\|_\infty \leq \epsilon_{\text{quant}}$ | $\sim 10^{-2}$ |
| Deploy → Serve | $G_q(x_{\text{API}}) = G_q(x_{\text{local}})$ | exact (bit-identical) |

### Pipeline Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                    END-TO-END ML PIPELINE WITH ONNX                             │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐  │
│  │  TRAIN   │    │  EXPORT  │    │ OPTIMIZE │    │ QUANTIZE │    │  DEPLOY  │  │
│  │          │    │          │    │          │    │          │    │          │  │
│  │ PyTorch  │───▶│ torch.   │───▶│ ORT/onnx │───▶│ INT8/    │───▶│ FastAPI  │  │
│  │ training │    │ onnx.    │    │ optimizer│    │ FP16     │    │ + ORT    │  │
│  │ loop     │    │ export   │    │ passes   │    │ calib.   │    │ session  │  │
│  └────┬─────┘    └────┬─────┘    └────┬─────┘    └────┬─────┘    └────┬─────┘  │
│       │               │               │               │               │         │
│       ▼               ▼               ▼               ▼               ▼         │
│  checkpoint.pt   model.onnx     model_opt.onnx  model_q.onnx    /predict       │
│  (state_dict)    (FP32 graph)   (fused graph)   (INT8 graph)    /health         │
│                                                                                 │
│  ◄──── Parity ────► ◄──── Parity ────► ◄──── Accuracy ──► ◄──── Monitor ──►   │
│       Check              Check              Check              SLOs             │
│                                                                                 │
│  ┌─────────────────────────────────────────────────────────────────────────┐    │
│  │                    CI/CD PIPELINE (automated)                           │    │
│  │  git push → train → export → test parity → optimize → benchmark       │    │
│  │  → quantize → validate accuracy → deploy canary → promote             │    │
│  └─────────────────────────────────────────────────────────────────────────┘    │
│                                                                                 │
│  ┌─────────────────────────────────────────────────────────────────────────┐    │
│  │                    MONITORING (continuous)                              │    │
│  │  latency p50/p95/p99 │ error rate │ drift detection │ accuracy decay   │    │
│  └─────────────────────────────────────────────────────────────────────────┘    │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
!pip install torch torchvision onnx onnxruntime numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import onnx
from onnx import numpy_helper
import onnxruntime as ort
import matplotlib.pyplot as plt
import time
import json
import hashlib
from collections import defaultdict
from pathlib import Path

print(f"PyTorch:      {torch.__version__}")
print(f"ONNX:         {onnx.__version__}")
print(f"ORT:          {ort.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
print(f"ORT providers: {ort.get_available_providers()}")

<a id='section-2'></a>
## Section 2: Training Phase — Loss, Optimization, Convergence

### Mathematical Framework

Training solves the empirical risk minimization (ERM) problem:

$$\theta^* = \arg\min_\theta \frac{1}{N} \sum_{i=1}^N \mathcal{L}\bigl(f_\theta(x_i),\; y_i\bigr) + \lambda \Omega(\theta)$$

where:
- $f_\theta: \mathbb{R}^{C \times H \times W} \to \mathbb{R}^K$ — parameterized model (CNN)
- $\mathcal{L}$ — cross-entropy loss: $\mathcal{L}(\hat{y}, y) = -\sum_{k=1}^K y_k \log \text{softmax}(\hat{y})_k$
- $\Omega(\theta) = \|\theta\|_2^2$ — weight decay (L2 regularization)
- $\lambda$ — regularization strength

### SGD with Momentum Update Rule

At each step $t$, given mini-batch $B_t$:

$$g_t = \frac{1}{|B_t|} \sum_{(x,y) \in B_t} \nabla_\theta \mathcal{L}(f_\theta(x), y)$$

$$v_t = \mu \cdot v_{t-1} + g_t + \lambda \theta_{t-1}$$

$$\theta_t = \theta_{t-1} - \eta \cdot v_t$$

where $\eta$ is the learning rate, $\mu$ is momentum (typically 0.9), and $\lambda$ is weight decay.

### Convergence Criterion

Training terminates when one of:
1. **Epoch limit**: $t > T_{\max}$ (e.g., 20 epochs)
2. **Loss plateau**: $|\mathcal{L}_t - \mathcal{L}_{t-k}| < \delta$ for patience $k$
3. **Validation metric**: accuracy on held-out set stops improving

### SmallCnn Architecture for CIFAR-10

```
Input: [B, 3, 32, 32]
  │
  ├──▶ Conv2d(3→32, 3×3, pad=1) ──▶ ReLU ──▶ MaxPool2d(2)
  │    Output: [B, 32, 16, 16]
  │
  ├──▶ Conv2d(32→64, 3×3, pad=1) ──▶ ReLU ──▶ MaxPool2d(2)
  │    Output: [B, 64, 8, 8]
  │
  ├──▶ Conv2d(64→128, 3×3, pad=1) ──▶ ReLU ──▶ MaxPool2d(2)
  │    Output: [B, 128, 4, 4]
  │
  ├──▶ Flatten ──▶ [B, 2048]
  │
  ├──▶ Linear(2048→256) ──▶ ReLU
  │
  └──▶ Linear(256→10) ──▶ Logits [B, 10]
```

### Parameter Count Analysis

| Layer | Input Shape | Params | FLOPs (per sample) |
|:---|:---|---:|---:|
| Conv1 (3→32, 3×3) | $3 \times 32 \times 32$ | $3 \cdot 32 \cdot 9 + 32 = 896$ | $896 \cdot 32 \cdot 32 = 917{,}504$ |
| Conv2 (32→64, 3×3) | $32 \times 16 \times 16$ | $32 \cdot 64 \cdot 9 + 64 = 18{,}496$ | $18{,}496 \cdot 16 \cdot 16 = 4{,}734{,}976$ |
| Conv3 (64→128, 3×3) | $64 \times 8 \times 8$ | $64 \cdot 128 \cdot 9 + 128 = 73{,}856$ | $73{,}856 \cdot 8 \cdot 8 = 4{,}726{,}784$ |
| FC1 (2048→256) | $2048$ | $2048 \cdot 256 + 256 = 524{,}544$ | $524{,}544$ |
| FC2 (256→10) | $256$ | $256 \cdot 10 + 10 = 2{,}570$ | $2{,}570$ |
| **Total** | | **620,362** | **~10.9M** |

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                'dog', 'frog', 'horse', 'ship', 'truck']


class SmallCnn(nn.Module):
    """CIFAR-10 CNN designed for CPU training feasibility."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


model = SmallCnn()
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SmallCnn architecture:")
print(f"  Total parameters:     {total_params:>10,}")
print(f"  Trainable parameters: {trainable:>10,}")
print(f"  Model size (FP32):    {total_params * 4 / 1024:.1f} KB")
print()

for name, p in model.named_parameters():
    print(f"  {name:<20} {str(list(p.shape)):<20} {p.numel():>8,} params")

x = torch.randn(1, 3, 32, 32)
with torch.no_grad():
    logits = model(x)
print(f"\nForward pass: {x.shape} → {logits.shape}")

In [ ]:
# Simulate a training loop with synthetic data to demonstrate the training phase
# (Use real CIFAR-10 via torchvision for actual training; this is for demonstration)

torch.manual_seed(42)
np.random.seed(42)

model = SmallCnn()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

n_samples = 512
n_epochs = 10
batch_size = 64

X_synth = torch.randn(n_samples, 3, 32, 32)
y_synth = torch.randint(0, 10, (n_samples,))

history = {'loss': [], 'accuracy': []}

model.train()
for epoch in range(n_epochs):
    epoch_loss = 0.0
    correct = 0
    total = 0

    perm = torch.randperm(n_samples)
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_synth[idx], y_synth[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(idx)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(idx)

    avg_loss = epoch_loss / total
    acc = correct / total
    history['loss'].append(avg_loss)
    history['accuracy'].append(acc)
    print(f"Epoch {epoch+1:>2}/{n_epochs}  loss={avg_loss:.4f}  acc={acc:.3f}")

print(f"\nFinal training loss: {history['loss'][-1]:.4f}")
print(f"Final training acc:  {history['accuracy'][-1]:.3f}")

In [ ]:
# Visualize training convergence
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(range(1, n_epochs + 1), history['loss'], 'o-', color='#e74c3c',
             linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Training Loss Convergence', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, n_epochs + 1), history['accuracy'], 's-', color='#2ecc71',
             linewidth=2, markersize=6)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Save checkpoint
checkpoint = {
    'state_dict': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'epoch': n_epochs,
    'loss': history['loss'][-1],
    'accuracy': history['accuracy'][-1],
    'cifar_mean': CIFAR_MEAN,
    'cifar_std': CIFAR_STD,
}
ckpt_path = '/tmp/e2e_checkpoint.pt'
torch.save(checkpoint, ckpt_path)
print(f"Checkpoint saved to {ckpt_path}")
print(f"Checkpoint size: {Path(ckpt_path).stat().st_size / 1024:.1f} KB")

<a id='section-3'></a>
## Section 3: ONNX Export — Graph Construction and Operator Mapping

### Export as a Graph Homomorphism

ONNX export is a mapping from the PyTorch computation graph $G_{\text{PT}}$ to an ONNX graph $G_{\text{ONNX}}$:

$$\mathcal{E}: G_{\text{PT}} = (V_{\text{PT}}, E_{\text{PT}}) \to G_{\text{ONNX}} = (V_{\text{ONNX}}, E_{\text{ONNX}})$$

This is a **structure-preserving** map where:
- Each PyTorch op $v \in V_{\text{PT}}$ maps to one or more ONNX ops in $V_{\text{ONNX}}$
- Data dependencies in $E_{\text{PT}}$ are preserved in $E_{\text{ONNX}}$
- Numerical semantics are preserved: $G_{\text{ONNX}}(x) \approx G_{\text{PT}}(x)$ for all valid $x$

### Operator Mapping Table

| PyTorch Operation | ONNX Operator(s) | Notes |
|:---|:---|:---|
| `nn.Conv2d` | `Conv` | kernel_shape, pads, strides as attributes |
| `F.relu` | `Relu` | Element-wise, in-place ignored |
| `nn.MaxPool2d` | `MaxPool` | kernel_shape, strides |
| `torch.flatten` | `Flatten` or `Reshape` | axis attribute |
| `nn.Linear` | `Gemm` or `MatMul + Add` | Depends on exporter version |
| `F.softmax` | `Softmax` | axis attribute |
| `F.cross_entropy` | N/A (training only) | Not exported for inference |

### Export Configuration

Key `torch.onnx.export` parameters:

```python
torch.onnx.export(
    model,                          # trained nn.Module
    dummy_input,                    # tensor matching input shape
    path,                           # output .onnx file
    input_names=['input'],          # named inputs for the graph
    output_names=['logits'],        # named outputs for the graph
    dynamic_axes={                  # which axes are variable-length
        'input': {0: 'batch'},
        'logits': {0: 'batch'},
    },
    opset_version=17,               # ONNX opset version
    do_constant_folding=True,       # fold constants during export
)
```

### Dynamic Axes

Without dynamic axes, the batch dimension is fixed to whatever `dummy_input` uses.
With `dynamic_axes={'input': {0: 'batch'}}`, the exported graph accepts any batch size:

$$\text{input} \in \mathbb{R}^{B \times 3 \times 32 \times 32}, \quad B \in \mathbb{Z}^+$$

In [ ]:
# ONNX Export
model.eval()

dummy_input = torch.randn(1, 3, 32, 32)
onnx_fp32_path = '/tmp/e2e_model_fp32.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_fp32_path,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)

# Validate the exported model
onnx_model = onnx.load(onnx_fp32_path)
onnx.checker.check_model(onnx_model)

print("ONNX Export Report")
print("=" * 55)
print(f"  File:         {onnx_fp32_path}")
print(f"  File size:    {Path(onnx_fp32_path).stat().st_size / 1024:.1f} KB")
print(f"  Opset:        {onnx_model.opset_import[0].version}")
print(f"  IR version:   {onnx_model.ir_version}")
print(f"  Nodes:        {len(onnx_model.graph.node)}")
print(f"  Initializers: {len(onnx_model.graph.initializer)}")
print(f"  Inputs:       {[(i.name, [d.dim_value or d.dim_param for d in i.type.tensor_type.shape.dim]) for i in onnx_model.graph.input if i.name not in [init.name for init in onnx_model.graph.initializer]]}")
print(f"  Outputs:      {[(o.name, [d.dim_value or d.dim_param for d in o.type.tensor_type.shape.dim]) for o in onnx_model.graph.output]}")
print(f"\nOperator histogram:")
op_counts = defaultdict(int)
for node in onnx_model.graph.node:
    op_counts[node.op_type] += 1
for op, count in sorted(op_counts.items()):
    print(f"    {op:<25} {count}")

# Compute model hash for versioning
with open(onnx_fp32_path, 'rb') as f:
    model_hash = hashlib.sha256(f.read()).hexdigest()[:16]
print(f"\n  SHA256 (first 16): {model_hash}")

<a id='section-4'></a>
## Section 4: Numerical Parity Verification

### Why Parity Testing is Critical

The export process introduces subtle numerical differences due to:
1. **Operator decomposition**: One PyTorch op may become multiple ONNX ops
2. **Floating-point evaluation order**: Different accumulation order → different rounding
3. **Constant folding**: Pre-computed values may round differently
4. **Library differences**: cuDNN vs ORT kernel implementations

### Parity Testing Protocol

For $n$ random test inputs $x_1, \ldots, x_n$:

$$\text{PASS} \iff \max_{i=1}^n \|f_\theta(x_i) - G_{\text{ONNX}}(x_i)\|_\infty < \epsilon_{\text{tol}}$$

where $\epsilon_{\text{tol}} = 10^{-5}$ for FP32.

### Statistical Parity Analysis

Beyond max-diff, we report:
- **Max absolute error**: worst-case divergence
- **Mean absolute error**: average divergence
- **Relative error**: $\frac{|y_{\text{PT}} - y_{\text{ORT}}|}{|y_{\text{PT}}| + \epsilon}$
- **Agreement rate**: fraction of samples with identical argmax predictions

In [ ]:
# Comprehensive parity testing: PyTorch vs ONNX Runtime

sess = ort.InferenceSession(onnx_fp32_path, providers=['CPUExecutionProvider'])
model.eval()

n_parity_tests = 500
batch_sizes = [1, 2, 4, 8, 16]

parity_results = {
    'max_abs_errors': [],
    'mean_abs_errors': [],
    'max_rel_errors': [],
    'prediction_matches': [],
}

torch.manual_seed(0)
for i in range(n_parity_tests):
    bs = batch_sizes[i % len(batch_sizes)]
    x_test = torch.randn(bs, 3, 32, 32)

    with torch.no_grad():
        y_pt = model(x_test).numpy()

    y_ort = sess.run(None, {'input': x_test.numpy()})[0]

    abs_err = np.abs(y_pt - y_ort)
    rel_err = abs_err / (np.abs(y_pt) + 1e-8)

    parity_results['max_abs_errors'].append(abs_err.max())
    parity_results['mean_abs_errors'].append(abs_err.mean())
    parity_results['max_rel_errors'].append(rel_err.max())
    parity_results['prediction_matches'].append(
        (y_pt.argmax(axis=1) == y_ort.argmax(axis=1)).all()
    )

max_abs = np.array(parity_results['max_abs_errors'])
mean_abs = np.array(parity_results['mean_abs_errors'])
max_rel = np.array(parity_results['max_rel_errors'])
pred_match = np.array(parity_results['prediction_matches'])

tol = 1e-5
all_pass = np.all(max_abs < tol)

print("╔════════════════════════════════════════════════════════════╗")
print("║         PyTorch vs ORT Parity Verification Report         ║")
print("╠════════════════════════════════════════════════════════════╣")
print(f"║  Test samples:          {n_parity_tests:>8}                         ║")
print(f"║  Batch sizes tested:    {batch_sizes}                    ║")
print(f"║  Tolerance (ε):         {tol:.0e}                         ║")
print("╠════════════════════════════════════════════════════════════╣")
print(f"║  Max |err| (worst):     {max_abs.max():.2e}                        ║")
print(f"║  Max |err| (median):    {np.median(max_abs):.2e}                        ║")
print(f"║  Mean |err| (avg):      {mean_abs.mean():.2e}                        ║")
print(f"║  Max relative error:    {max_rel.max():.2e}                        ║")
print(f"║  Prediction agreement:  {pred_match.mean()*100:.1f}%                          ║")
print(f"║  ALL within tolerance:  {'PASS' if all_pass else 'FAIL':>5}                          ║")
print("╚════════════════════════════════════════════════════════════╝")

In [ ]:
# Visualize parity error distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(max_abs, bins=40, color='#3498db', edgecolor='black', alpha=0.85)
axes[0].axvline(tol, color='red', linestyle='--', linewidth=2,
                label=f'Tolerance ε={tol:.0e}')
axes[0].set_xlabel('Max Absolute Error', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Max |PyTorch − ORT| per Sample', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].hist(mean_abs, bins=40, color='#2ecc71', edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Mean Absolute Error', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Mean |PyTorch − ORT| per Sample', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

axes[2].hist(max_rel, bins=40, color='#e74c3c', edgecolor='black', alpha=0.85)
axes[2].set_xlabel('Max Relative Error', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Max Relative Error per Sample', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-5'></a>
## Section 5: Graph Optimization Pipeline

### Optimization as Semantic-Preserving Graph Rewriting

Graph optimization applies a sequence of rewrite rules $R_1, R_2, \ldots, R_m$ to the exported graph:

$$G' = R_m \circ \cdots \circ R_2 \circ R_1(G)$$

Each rule $R_i$ must satisfy **semantic preservation**:

$$\forall x \in \mathcal{X}: \|R_i(G)(x) - G(x)\|_\infty \leq \epsilon_{\text{fp}}$$

### Key Optimization Passes

**1. Constant Folding (Partial Evaluation)**

Evaluate subgraphs with all-constant inputs at optimization time:

$$\text{PE}(G, \sigma) \to G' \quad \text{where } |V_{G'}| \leq |V_G|$$

**2. Operator Fusion**

Combine sequences of ops into single fused kernels. The key benefit is eliminating
intermediate memory traffic. For $k$ fused element-wise ops:

$$\text{Memory savings} = (k-1) \times 2 \times |T| \times \text{sizeof(dtype)}$$

**3. Dead Code Elimination**

Remove nodes whose outputs are never consumed by any graph output.

### ORT Optimization Levels

| Level | Constant | Passes |
|:---:|:---|:---|
| 0 | `ORT_DISABLE_ALL` | None |
| 1 | `ORT_ENABLE_BASIC` | Constant folding, identity removal, shape inference |
| 2 | `ORT_ENABLE_EXTENDED` | + Conv+BN fusion, MatMul+Add→Gemm, attention fusion |
| 99 | `ORT_ENABLE_ALL` | + Layout transforms, all EP-specific optimizations |

In [ ]:
# Apply graph optimizations and compare across levels

onnx_model = onnx.load(onnx_fp32_path)
original_node_count = len(onnx_model.graph.node)

optimization_results = {}
levels = [
    ('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ('BASIC', ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ('ALL', ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

onnx_optimized_path = '/tmp/e2e_model_optimized.onnx'

print(f"Original graph: {original_node_count} nodes")
print(f"Op types: {[n.op_type for n in onnx_model.graph.node]}")
print("\n" + "=" * 60)

for name, level in levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    outpath = f'/tmp/e2e_opt_{name}.onnx'
    so.optimized_model_filepath = outpath
    sess_opt = ort.InferenceSession(onnx_fp32_path, so,
                                    providers=['CPUExecutionProvider'])
    opt_model = onnx.load(outpath)
    n_nodes = len(opt_model.graph.node)
    op_types = [n.op_type for n in opt_model.graph.node]
    optimization_results[name] = {
        'nodes': n_nodes,
        'ops': op_types,
        'path': outpath,
    }
    reduction = original_node_count - n_nodes
    print(f"\n{name} (level={level.numerator}): {n_nodes} nodes (−{reduction})")
    for op in sorted(set(op_types)):
        print(f"    {op}: {op_types.count(op)}")

# Use the fully optimized model going forward
best_opt_path = optimization_results['ALL']['path']
print(f"\nUsing fully optimized model: {best_opt_path}")

<a id='section-6'></a>
## Section 6: Quantization Pipeline — Calibration and Error Analysis

### Quantization as Precision-Accuracy Tradeoff

Quantization maps FP32 tensors to lower-precision (INT8) representations:

$$Q(r) = \text{round}\left(\frac{r}{s}\right) + z$$

where:
- $r \in \mathbb{R}$ — real-valued (FP32) tensor element
- $s > 0$ — **scale** (maps real range to integer range)
- $z \in \mathbb{Z}$ — **zero point** (maps real zero to an integer)
- $Q(r) \in [0, 255]$ for uint8 or $[-128, 127]$ for int8

Dequantization recovers the approximation:

$$\hat{r} = s \cdot (Q(r) - z) \approx r$$

### Quantization Error

The quantization error per element is bounded by:

$$|r - \hat{r}| \leq \frac{s}{2}$$

where $s = \frac{r_{\max} - r_{\min}}{2^b - 1}$ for $b$-bit quantization.

### Dynamic vs Static Quantization

| Property | Dynamic | Static |
|:---|:---|:---|
| **Weights** | Quantized offline | Quantized offline |
| **Activations** | Quantized at runtime per-batch | Quantized with pre-calibrated scales |
| **Calibration data** | Not required | Required (representative dataset) |
| **Accuracy** | Generally higher | Can be lower but faster |
| **Best for** | Models where activation range varies | Stable activation distributions |

### Expected Impact

$$\text{Model size} \approx \frac{\text{FP32 size}}{4} \times (1 + \text{overhead})$$

$$\text{Latency improvement} \approx 1.2\text{–}3\times \text{ on CPU (GEMM-bound layers)}$$

$$\text{Accuracy degradation} \approx 0.1\text{–}2\% \text{ (task-dependent)}$$

In [ ]:
# Dynamic INT8 quantization
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_int8_path = '/tmp/e2e_model_int8.onnx'

quantize_dynamic(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_path,
    weight_type=QuantType.QUInt8,
)

fp32_size = Path(onnx_fp32_path).stat().st_size
int8_size = Path(onnx_int8_path).stat().st_size

q_model = onnx.load(onnx_int8_path)

print("Quantization Report")
print("=" * 55)
print(f"  FP32 model size:   {fp32_size / 1024:.1f} KB")
print(f"  INT8 model size:   {int8_size / 1024:.1f} KB")
print(f"  Compression ratio: {fp32_size / int8_size:.2f}×")
print(f"  FP32 nodes:        {len(onnx_model.graph.node)}")
print(f"  INT8 nodes:        {len(q_model.graph.node)}")

q_op_counts = defaultdict(int)
for node in q_model.graph.node:
    q_op_counts[node.op_type] += 1
print(f"\n  INT8 operator histogram:")
for op, count in sorted(q_op_counts.items()):
    marker = " ← quantized" if 'Quantize' in op or 'Integer' in op or 'QLinear' in op or op.startswith('DynamicQuantize') else ""
    print(f"    {op:<30} {count}{marker}")

In [ ]:
# Quantization accuracy comparison: FP32 vs INT8

sess_fp32 = ort.InferenceSession(onnx_fp32_path, providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession(onnx_int8_path, providers=['CPUExecutionProvider'])

n_quant_tests = 500
quant_abs_errors = []
quant_rel_errors = []
quant_prediction_matches = []
quant_confidence_diffs = []

torch.manual_seed(123)
for i in range(n_quant_tests):
    x_test = torch.randn(1, 3, 32, 32).numpy()

    y_fp32 = sess_fp32.run(None, {'input': x_test})[0]
    y_int8 = sess_int8.run(None, {'input': x_test})[0]

    abs_err = np.abs(y_fp32 - y_int8)
    rel_err = abs_err / (np.abs(y_fp32) + 1e-8)

    quant_abs_errors.append(abs_err.max())
    quant_rel_errors.append(rel_err.max())
    quant_prediction_matches.append(
        y_fp32.argmax(axis=1)[0] == y_int8.argmax(axis=1)[0]
    )

    softmax_fp32 = np.exp(y_fp32) / np.exp(y_fp32).sum(axis=1, keepdims=True)
    softmax_int8 = np.exp(y_int8) / np.exp(y_int8).sum(axis=1, keepdims=True)
    quant_confidence_diffs.append(np.abs(softmax_fp32.max() - softmax_int8.max()))

quant_abs = np.array(quant_abs_errors)
quant_rel = np.array(quant_rel_errors)
quant_match = np.array(quant_prediction_matches)
quant_conf = np.array(quant_confidence_diffs)

print("╔════════════════════════════════════════════════════════════╗")
print("║       FP32 vs INT8 Quantization Accuracy Report           ║")
print("╠════════════════════════════════════════════════════════════╣")
print(f"║  Test samples:               {n_quant_tests:>8}                    ║")
print(f"║  Max |logit diff| (worst):   {quant_abs.max():.4f}                     ║")
print(f"║  Max |logit diff| (median):  {np.median(quant_abs):.4f}                     ║")
print(f"║  Mean |logit diff|:          {quant_abs.mean():.4f}                     ║")
print(f"║  Prediction agreement:       {quant_match.mean()*100:.1f}%                       ║")
print(f"║  Mean confidence shift:      {quant_conf.mean():.4f}                     ║")
print("╚════════════════════════════════════════════════════════════╝")

In [ ]:
# Visualize quantization impact
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(quant_abs, bins=40, color='#e74c3c', edgecolor='black', alpha=0.85)
axes[0].set_xlabel('Max |FP32 − INT8| Logit Difference', fontsize=10)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Quantization Error Distribution', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

categories = ['FP32', 'INT8']
sizes = [fp32_size / 1024, int8_size / 1024]
bars = axes[1].bar(categories, sizes, color=['#3498db', '#e67e22'],
                   edgecolor='black', linewidth=1.2, width=0.5)
axes[1].set_ylabel('Size (KB)', fontsize=11)
axes[1].set_title('Model Size Comparison', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for bar, s in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{s:.0f} KB', ha='center', fontweight='bold', fontsize=11)

axes[2].hist(quant_conf, bins=40, color='#9b59b6', edgecolor='black', alpha=0.85)
axes[2].set_xlabel('|Confidence FP32 − Confidence INT8|', fontsize=10)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Prediction Confidence Shift', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7: Deployment Architecture

### Serving Patterns

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    DEPLOYMENT ARCHITECTURE                              │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Pattern 1: Embedded Inference (Single Process)                        │
│  ┌─────────────────────────────────────────┐                           │
│  │ FastAPI / Flask                          │                           │
│  │ ┌─────────────────────────────────────┐  │                           │
│  │ │ ORT InferenceSession (in-process)   │  │                           │
│  │ │ model.onnx loaded at startup        │  │                           │
│  │ └─────────────────────────────────────┘  │                           │
│  └─────────────────────────────────────────┘                           │
│  + Simple, low latency                                                  │
│  − Coupled scaling (API + model share resources)                       │
│                                                                         │
│  Pattern 2: Model Server (Decoupled)                                   │
│  ┌──────────────┐    gRPC     ┌──────────────────┐                     │
│  │ API Gateway   │──────────▶│ Triton / ORT     │                     │
│  │ (routing,     │           │ Inference Server │                     │
│  │  auth, rate   │◀──────────│ (model.onnx)     │                     │
│  │  limiting)    │           └──────────────────┘                     │
│  └──────────────┘                                                      │
│  + Independent scaling, model hot-swap                                 │
│  − Network hop latency, operational complexity                         │
│                                                                         │
│  Pattern 3: Serverless (Event-Driven)                                  │
│  ┌──────────┐   event   ┌───────────────────┐                          │
│  │ API GW / │─────────▶│ Lambda / Cloud Run │                          │
│  │ Queue    │          │ + ORT (cold start!) │                          │
│  └──────────┘          └───────────────────┘                          │
│  + Zero cost at idle, auto-scaling                                     │
│  − Cold start latency (model load), memory limits                     │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### API Design for ML Serving

A production ML API should expose:

| Endpoint | Method | Purpose |
|:---|:---|:---|
| `/health` | GET | Liveness + model metadata (hash, providers, input/output shapes) |
| `/predict` | POST | Inference on raw tensor (JSON body) |
| `/predict_file` | POST | Inference on image file (multipart form) |
| `/metrics` | GET | Prometheus-compatible metrics (latency, throughput, errors) |

### Containerization

```dockerfile
FROM python:3.10-slim
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY artifacts/ /app/artifacts/
COPY deploy_api.py /app/
WORKDIR /app
EXPOSE 9000
CMD ["uvicorn", "deploy_api:app", "--host", "0.0.0.0", "--port", "9000"]
```

### Request Processing Pipeline

```
Client Request ──▶ Validate Input ──▶ Preprocess (normalize) ──▶ ORT.run()
                                                                    │
Client Response ◀── Format Output ◀── Postprocess (argmax/softmax) ◀┘
```

In [ ]:
# Simulate a production serving endpoint (without actually starting a server)

class ONNXModelServer:
    """Minimal model server wrapping an ORT session."""

    def __init__(self, model_path: str):
        self.model_path = model_path
        self.session = ort.InferenceSession(
            model_path, providers=['CPUExecutionProvider']
        )
        with open(model_path, 'rb') as f:
            self.model_hash = hashlib.sha256(f.read()).hexdigest()[:16]

        self.input_meta = [
            {'name': i.name, 'shape': i.shape, 'type': i.type}
            for i in self.session.get_inputs()
        ]
        self.output_meta = [
            {'name': o.name, 'shape': o.shape, 'type': o.type}
            for o in self.session.get_outputs()
        ]
        self.request_count = 0
        self.latencies = []

    def health(self) -> dict:
        return {
            'status': 'healthy',
            'model_path': self.model_path,
            'model_hash': self.model_hash,
            'providers': self.session.get_providers(),
            'inputs': self.input_meta,
            'outputs': self.output_meta,
            'requests_served': self.request_count,
        }

    def predict(self, input_tensor: np.ndarray) -> dict:
        if input_tensor.ndim != 4 or input_tensor.shape[1:] != (3, 32, 32):
            raise ValueError(f"Expected shape (B, 3, 32, 32), got {input_tensor.shape}")

        t0 = time.perf_counter()
        logits = self.session.run(None, {'input': input_tensor})[0]
        latency_ms = (time.perf_counter() - t0) * 1000

        self.request_count += 1
        self.latencies.append(latency_ms)

        probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        predictions = logits.argmax(axis=1)

        return {
            'predictions': [CIFAR_LABELS[p] for p in predictions],
            'class_indices': predictions.tolist(),
            'confidences': [float(probs[i, predictions[i]]) for i in range(len(predictions))],
            'latency_ms': round(latency_ms, 3),
        }


# Instantiate and test
server = ONNXModelServer(onnx_fp32_path)

print("Health check:")
health = server.health()
print(json.dumps(health, indent=2, default=str))

print("\nPrediction test:")
x_test = np.random.randn(4, 3, 32, 32).astype(np.float32)
result = server.predict(x_test)
for i, (label, conf) in enumerate(zip(result['predictions'], result['confidences'])):
    print(f"  Sample {i}: {label} (confidence: {conf:.3f})")
print(f"  Latency: {result['latency_ms']:.3f} ms")

<a id='section-8'></a>
## Section 8: Benchmarking Framework

### Latency Measurement Methodology

Proper benchmarking requires:

1. **Warmup**: First $W$ iterations are discarded (JIT compilation, cache warming)
2. **Steady-state measurement**: $N$ iterations timed after warmup
3. **Statistical reporting**: Report percentiles, not just mean

$$\text{Reported metrics: } p_{50}, p_{95}, p_{99}, \mu, \sigma$$

### Statistical Significance

To determine if optimization $A$ is faster than baseline $B$, use a paired $t$-test:

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad d_i = T_B^{(i)} - T_A^{(i)}$$

where $\bar{d}$ is the mean of paired differences and $s_d$ is their standard deviation.

Reject $H_0: \mu_d = 0$ if $|t| > t_{\alpha/2, n-1}$ (typically $\alpha = 0.05$).

### Throughput Calculation

$$\text{Throughput (samples/sec)} = \frac{\text{batch\_size}}{\text{latency (sec)}}$$

### Benchmark Design Principles

```
┌──────────────────────────────────────────────────────────┐
│                 BENCHMARKING PROTOCOL                     │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  1. Pin CPU frequency (disable turbo boost)              │
│  2. Set OMP_NUM_THREADS / intra_op_num_threads           │
│  3. Warmup: 50–100 iterations (discard)                  │
│  4. Measure: 1000+ iterations (record each)              │
│  5. Report: p50, p95, p99, mean, std                     │
│  6. Repeat 3× with different random inputs               │
│  7. Statistical test for significance                    │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# Comprehensive benchmarking: FP32 vs INT8 across batch sizes

def benchmark_session(session, input_name, input_shape, n_warmup=100, n_runs=1000):
    """Benchmark an ORT session with proper warmup and statistical reporting."""
    x = np.random.randn(*input_shape).astype(np.float32)

    for _ in range(n_warmup):
        session.run(None, {input_name: x})

    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(None, {input_name: x})
        latencies.append((time.perf_counter() - t0) * 1000)

    lat = np.array(latencies)
    return {
        'mean': lat.mean(),
        'std': lat.std(),
        'p50': np.percentile(lat, 50),
        'p95': np.percentile(lat, 95),
        'p99': np.percentile(lat, 99),
        'min': lat.min(),
        'max': lat.max(),
        'throughput': input_shape[0] / (lat.mean() / 1000),
        'raw': lat,
    }


sess_fp32 = ort.InferenceSession(onnx_fp32_path, providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession(onnx_int8_path, providers=['CPUExecutionProvider'])

batch_sizes_bench = [1, 4, 16, 32]
bench_results = {'fp32': {}, 'int8': {}}

print(f"{'Batch':>6} {'Model':>6} {'Mean(ms)':>10} {'p50(ms)':>10} {'p95(ms)':>10} "
      f"{'p99(ms)':>10} {'Throughput':>12}")
print("─" * 72)

for bs in batch_sizes_bench:
    shape = (bs, 3, 32, 32)
    for label, sess in [('fp32', sess_fp32), ('int8', sess_int8)]:
        r = benchmark_session(sess, 'input', shape, n_warmup=50, n_runs=500)
        bench_results[label][bs] = r
        print(f"{bs:>6} {label:>6} {r['mean']:>10.3f} {r['p50']:>10.3f} "
              f"{r['p95']:>10.3f} {r['p99']:>10.3f} {r['throughput']:>10.1f} s/s")

In [ ]:
# Visualize benchmark results
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

x_pos = np.arange(len(batch_sizes_bench))
width = 0.35

# Latency comparison (p50)
fp32_p50 = [bench_results['fp32'][bs]['p50'] for bs in batch_sizes_bench]
int8_p50 = [bench_results['int8'][bs]['p50'] for bs in batch_sizes_bench]
axes[0].bar(x_pos - width/2, fp32_p50, width, label='FP32', color='#3498db',
            edgecolor='black')
axes[0].bar(x_pos + width/2, int8_p50, width, label='INT8', color='#e67e22',
            edgecolor='black')
axes[0].set_xlabel('Batch Size', fontsize=11)
axes[0].set_ylabel('p50 Latency (ms)', fontsize=11)
axes[0].set_title('Median Latency: FP32 vs INT8', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(batch_sizes_bench)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Throughput comparison
fp32_tp = [bench_results['fp32'][bs]['throughput'] for bs in batch_sizes_bench]
int8_tp = [bench_results['int8'][bs]['throughput'] for bs in batch_sizes_bench]
axes[1].bar(x_pos - width/2, fp32_tp, width, label='FP32', color='#3498db',
            edgecolor='black')
axes[1].bar(x_pos + width/2, int8_tp, width, label='INT8', color='#e67e22',
            edgecolor='black')
axes[1].set_xlabel('Batch Size', fontsize=11)
axes[1].set_ylabel('Throughput (samples/sec)', fontsize=11)
axes[1].set_title('Throughput: FP32 vs INT8', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(batch_sizes_bench)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Latency distribution (batch_size=1)
lat_fp32 = bench_results['fp32'][1]['raw']
lat_int8 = bench_results['int8'][1]['raw']
axes[2].hist(lat_fp32, bins=50, alpha=0.7, label='FP32', color='#3498db',
             edgecolor='black')
axes[2].hist(lat_int8, bins=50, alpha=0.7, label='INT8', color='#e67e22',
             edgecolor='black')
axes[2].set_xlabel('Latency (ms)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Latency Distribution (batch=1)', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Speedup summary
print("\nINT8 Speedup over FP32 (p50 latency):")
for bs in batch_sizes_bench:
    speedup = bench_results['fp32'][bs]['p50'] / bench_results['int8'][bs]['p50']
    print(f"  Batch {bs:>3}: {speedup:.2f}×")

<a id='section-9'></a>
## Section 9: Monitoring and Observability

### What to Monitor in Production

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    ML MONITORING STACK                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Layer 1: Infrastructure Metrics                                       │
│  ├── CPU utilization, memory RSS, GPU utilization                      │
│  ├── Request rate, error rate (4xx/5xx)                                │
│  └── Container health, pod restarts                                    │
│                                                                         │
│  Layer 2: Model Performance Metrics                                    │
│  ├── Inference latency (p50, p95, p99)                                │
│  ├── Throughput (samples/sec)                                          │
│  └── Batch size distribution                                           │
│                                                                         │
│  Layer 3: Model Quality Metrics                                        │
│  ├── Prediction distribution (class balance over time)                 │
│  ├── Confidence distribution (shift → potential drift)                 │
│  ├── Feature drift (input distribution shift)                          │
│  └── Ground truth accuracy (when labels arrive)                        │
│                                                                         │
│  Layer 4: Business Metrics                                             │
│  ├── User engagement with predictions                                  │
│  ├── Conversion rate, revenue impact                                   │
│  └── Customer satisfaction scores                                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Drift Detection

**Data drift** occurs when the input distribution changes: $P_{\text{prod}}(x) \neq P_{\text{train}}(x)$.

Detect via Population Stability Index (PSI):

$$\text{PSI} = \sum_{i=1}^{B} (p_i - q_i) \cdot \ln\frac{p_i}{q_i}$$

where $p_i, q_i$ are bin proportions for production and training distributions.

| PSI Value | Interpretation |
|:---:|:---|
| $< 0.1$ | No significant drift |
| $0.1 - 0.2$ | Moderate drift — investigate |
| $> 0.2$ | Significant drift — retrain |

### SLO Definitions

| SLO | Target | Alert Threshold |
|:---|:---|:---|
| p99 latency | < 50 ms | > 75 ms for 5 min |
| Error rate | < 0.1% | > 1% for 2 min |
| Availability | 99.9% | < 99.5% in 1-hour window |
| Prediction confidence | Mean > 0.7 | Mean < 0.5 for 10 min |

In [ ]:
# Simulate production monitoring: drift detection and SLO tracking

def compute_psi(reference: np.ndarray, production: np.ndarray, n_bins: int = 10):
    """Compute Population Stability Index between two distributions."""
    min_val = min(reference.min(), production.min())
    max_val = max(reference.max(), production.max())
    bins = np.linspace(min_val, max_val, n_bins + 1)

    ref_hist, _ = np.histogram(reference, bins=bins)
    prod_hist, _ = np.histogram(production, bins=bins)

    ref_pct = (ref_hist + 1) / (len(reference) + n_bins)
    prod_pct = (prod_hist + 1) / (len(production) + n_bins)

    psi = np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct))
    return psi


np.random.seed(42)

# Simulate training distribution (reference)
ref_confidences = np.random.beta(5, 2, size=1000)
ref_latencies = np.random.lognormal(mean=np.log(2), sigma=0.3, size=1000)

# Simulate production windows (some with drift)
windows = {
    'Week 1 (normal)': {
        'confidences': np.random.beta(5, 2, size=1000),
        'latencies': np.random.lognormal(mean=np.log(2), sigma=0.3, size=1000),
    },
    'Week 2 (slight drift)': {
        'confidences': np.random.beta(4, 2.5, size=1000),
        'latencies': np.random.lognormal(mean=np.log(2.5), sigma=0.35, size=1000),
    },
    'Week 3 (major drift)': {
        'confidences': np.random.beta(2, 3, size=1000),
        'latencies': np.random.lognormal(mean=np.log(4), sigma=0.5, size=1000),
    },
    'Week 4 (after retrain)': {
        'confidences': np.random.beta(5.5, 1.8, size=1000),
        'latencies': np.random.lognormal(mean=np.log(2), sigma=0.3, size=1000),
    },
}

# SLO thresholds
SLO_P99_LATENCY = 50.0  # ms
SLO_MEAN_CONFIDENCE = 0.5
PSI_ALERT_THRESHOLD = 0.2

print("Production Monitoring Dashboard")
print("=" * 72)
print(f"{'Window':<25} {'Conf PSI':>10} {'Lat p99':>10} {'Mean Conf':>10} {'Status':>10}")
print("─" * 72)

for window_name, data in windows.items():
    conf_psi = compute_psi(ref_confidences, data['confidences'])
    lat_p99 = np.percentile(data['latencies'], 99)
    mean_conf = data['confidences'].mean()

    violations = []
    if conf_psi > PSI_ALERT_THRESHOLD:
        violations.append('DRIFT')
    if lat_p99 > SLO_P99_LATENCY:
        violations.append('LATENCY')
    if mean_conf < SLO_MEAN_CONFIDENCE:
        violations.append('CONF')

    status = ', '.join(violations) if violations else 'OK'
    print(f"{window_name:<25} {conf_psi:>10.4f} {lat_p99:>10.2f} {mean_conf:>10.3f} {status:>10}")

In [ ]:
# Visualize monitoring data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

window_names = list(windows.keys())
colors_w = ['#2ecc71', '#f39c12', '#e74c3c', '#3498db']

# Confidence distributions over time
for i, (name, data) in enumerate(windows.items()):
    axes[0, 0].hist(data['confidences'], bins=30, alpha=0.5, label=name,
                    color=colors_w[i], edgecolor='black', linewidth=0.5)
axes[0, 0].axvline(SLO_MEAN_CONFIDENCE, color='red', linestyle='--', linewidth=2,
                    label=f'SLO threshold ({SLO_MEAN_CONFIDENCE})')
axes[0, 0].set_xlabel('Prediction Confidence', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Confidence Distribution Over Time', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# PSI over time
psi_values = [compute_psi(ref_confidences, windows[w]['confidences']) for w in window_names]
bars = axes[0, 1].bar(range(len(window_names)), psi_values, color=colors_w,
                       edgecolor='black')
axes[0, 1].axhline(0.1, color='orange', linestyle='--', label='Investigate (0.1)')
axes[0, 1].axhline(PSI_ALERT_THRESHOLD, color='red', linestyle='--', label=f'Alert ({PSI_ALERT_THRESHOLD})')
axes[0, 1].set_ylabel('PSI', fontsize=11)
axes[0, 1].set_title('Population Stability Index', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(range(len(window_names)))
axes[0, 1].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)

# Latency distributions
for i, (name, data) in enumerate(windows.items()):
    axes[1, 0].hist(data['latencies'], bins=30, alpha=0.5, label=name,
                    color=colors_w[i], edgecolor='black', linewidth=0.5)
axes[1, 0].axvline(SLO_P99_LATENCY, color='red', linestyle='--', linewidth=2,
                    label=f'SLO p99 ({SLO_P99_LATENCY}ms)')
axes[1, 0].set_xlabel('Latency (ms)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Latency Distribution Over Time', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# SLO compliance summary
slo_metrics = ['p99 Lat', 'Conf PSI', 'Mean Conf']
compliance = np.array([
    [1, 1, 0, 1],
    [1, 1, 0, 1],
    [1, 1, 0, 1],
], dtype=float)
for j, (name, data) in enumerate(windows.items()):
    if np.percentile(data['latencies'], 99) > SLO_P99_LATENCY:
        compliance[0, j] = 0
    if compute_psi(ref_confidences, data['confidences']) > PSI_ALERT_THRESHOLD:
        compliance[1, j] = 0
    if data['confidences'].mean() < SLO_MEAN_CONFIDENCE:
        compliance[2, j] = 0

im = axes[1, 1].imshow(compliance, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1, 1].set_xticks(range(4))
axes[1, 1].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
axes[1, 1].set_yticks(range(3))
axes[1, 1].set_yticklabels(slo_metrics)
axes[1, 1].set_title('SLO Compliance Matrix', fontsize=12, fontweight='bold')
for i in range(3):
    for j in range(4):
        label = 'PASS' if compliance[i, j] else 'FAIL'
        axes[1, 1].text(j, i, label, ha='center', va='center',
                        fontweight='bold', fontsize=10,
                        color='white' if compliance[i, j] == 0 else 'black')

plt.tight_layout()
plt.show()

<a id='section-10'></a>
## Section 10: Model Versioning and Registry Patterns

### Model Registry Design

A model registry tracks all model artifacts with metadata for reproducibility and rollback.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       MODEL REGISTRY                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐    │
│  │ Model: cifar10-smallcnn                                        │    │
│  ├─────────────────────────────────────────────────────────────────┤    │
│  │                                                                 │    │
│  │  v1.0.0 (2024-01-15)    ◄── production (70% traffic)          │    │
│  │  ├── artifacts/model_v1.onnx  (sha256: a3f8...)               │    │
│  │  ├── metrics: acc=91.2%, p99_lat=12ms                         │    │
│  │  └── training: lr=0.01, epochs=50, data_hash=b7c2...          │    │
│  │                                                                 │    │
│  │  v1.1.0 (2024-02-01)    ◄── canary (30% traffic)             │    │
│  │  ├── artifacts/model_v1.1.onnx  (sha256: f2d1...)             │    │
│  │  ├── metrics: acc=92.0%, p99_lat=11ms                         │    │
│  │  └── training: lr=0.005, epochs=80, data_hash=b7c2...        │    │
│  │                                                                 │    │
│  │  v0.9.0 (2023-12-01)    ◄── archived (rollback candidate)    │    │
│  │  ├── artifacts/model_v0.9.onnx  (sha256: e1a5...)             │    │
│  │  └── metrics: acc=89.5%, p99_lat=14ms                         │    │
│  │                                                                 │    │
│  └─────────────────────────────────────────────────────────────────┘    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### A/B Testing Framework

A/B testing compares model versions in production with statistical rigor:

$$H_0: \mu_A = \mu_B \quad \text{vs} \quad H_1: \mu_A \neq \mu_B$$

**Sample size** for detecting a minimum effect size $\delta$ with power $1 - \beta$:

$$n = \frac{2(z_{\alpha/2} + z_\beta)^2 \sigma^2}{\delta^2}$$

For accuracy comparison ($\alpha=0.05, \beta=0.2, \sigma=0.05, \delta=0.01$):

$$n = \frac{2(1.96 + 0.84)^2 (0.05)^2}{(0.01)^2} \approx 3{,}920 \text{ samples per group}$$

### Rollback Strategy

```
Rollback trigger conditions:
  1. p99 latency > 2× baseline for > 5 minutes
  2. Error rate > 1% for > 2 minutes
  3. Accuracy drop > 3% on shadow evaluation
  4. Manual operator trigger

Rollback procedure:
  1. Switch traffic to previous stable version (instant via load balancer)
  2. Keep failed version for debugging (do not delete)
  3. Alert on-call engineer
  4. Create incident report with root cause analysis
```

In [ ]:
# Simulate a model registry and A/B testing scenario

class ModelRegistry:
    """Minimal model registry for tracking ONNX model versions."""

    def __init__(self):
        self.models = {}
        self.production_version = None
        self.canary_version = None

    def register(self, name: str, version: str, path: str, metrics: dict):
        with open(path, 'rb') as f:
            sha = hashlib.sha256(f.read()).hexdigest()[:16]
        key = f"{name}:{version}"
        self.models[key] = {
            'name': name, 'version': version, 'path': path,
            'sha256': sha, 'metrics': metrics,
            'size_kb': Path(path).stat().st_size / 1024,
        }
        return key

    def promote(self, key: str, stage: str):
        if stage == 'production':
            self.production_version = key
        elif stage == 'canary':
            self.canary_version = key

    def list_versions(self):
        for key, info in self.models.items():
            stage = ''
            if key == self.production_version:
                stage = ' ◄── PRODUCTION'
            elif key == self.canary_version:
                stage = ' ◄── CANARY'
            print(f"  {key}{stage}")
            print(f"    sha256: {info['sha256']}")
            print(f"    size:   {info['size_kb']:.1f} KB")
            print(f"    metrics: {info['metrics']}")


registry = ModelRegistry()

k1 = registry.register('cifar10-cnn', 'v1.0.0', onnx_fp32_path,
                        metrics={'accuracy': 0.912, 'p99_lat_ms': 12.3})
k2 = registry.register('cifar10-cnn', 'v1.1.0-int8', onnx_int8_path,
                        metrics={'accuracy': 0.905, 'p99_lat_ms': 8.1})

registry.promote(k1, 'production')
registry.promote(k2, 'canary')

print("Model Registry")
print("=" * 55)
registry.list_versions()

# Simulate A/B test
print("\nA/B Test Simulation (1000 requests, 70/30 split)")
print("─" * 55)

np.random.seed(42)
n_requests = 1000
group_a_correct = np.random.binomial(1, 0.912, size=int(n_requests * 0.7))
group_b_correct = np.random.binomial(1, 0.905, size=int(n_requests * 0.3))

acc_a = group_a_correct.mean()
acc_b = group_b_correct.mean()

from scipy import stats
t_stat, p_value = stats.ttest_ind(group_a_correct, group_b_correct)

print(f"  Model A (FP32):  acc={acc_a:.3f} (n={len(group_a_correct)})")
print(f"  Model B (INT8):  acc={acc_b:.3f} (n={len(group_b_correct)})")
print(f"  Δ accuracy:      {acc_a - acc_b:+.3f}")
print(f"  t-statistic:     {t_stat:.3f}")
print(f"  p-value:         {p_value:.4f}")
print(f"  Significant (α=0.05): {'Yes' if p_value < 0.05 else 'No'}")

if p_value >= 0.05:
    print("\n  → No significant accuracy difference. Safe to promote INT8 model.")
    print("    INT8 benefits: smaller model, potentially lower latency.")

<a id='section-11'></a>
## Section 11: CI/CD for ML Pipelines

### Automated Pipeline Stages

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    CI/CD PIPELINE FOR ML                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  git push ──▶ CI Trigger                                               │
│                  │                                                       │
│                  ▼                                                       │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │ Stage 1: TRAIN                                               │       │
│  │ ├── Reproduce training from config (deterministic seeds)     │       │
│  │ ├── Gate: loss < threshold, accuracy > minimum               │       │
│  │ └── Artifact: checkpoint.pt                                  │       │
│  └────────────────────────┬─────────────────────────────────────┘       │
│                           ▼                                             │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │ Stage 2: EXPORT + VALIDATE                                   │       │
│  │ ├── torch.onnx.export with fixed opset                      │       │
│  │ ├── onnx.checker.check_model (structural validity)          │       │
│  │ ├── Gate: parity test max|diff| < 1e-5                      │       │
│  │ └── Artifact: model.onnx                                    │       │
│  └────────────────────────┬─────────────────────────────────────┘       │
│                           ▼                                             │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │ Stage 3: OPTIMIZE + QUANTIZE                                 │       │
│  │ ├── Apply graph optimizations (ORT_ENABLE_ALL)              │       │
│  │ ├── Dynamic INT8 quantization                                │       │
│  │ ├── Gate: accuracy drop < 2%, latency improved              │       │
│  │ └── Artifacts: model_opt.onnx, model_int8.onnx              │       │
│  └────────────────────────┬─────────────────────────────────────┘       │
│                           ▼                                             │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │ Stage 4: BENCHMARK                                           │       │
│  │ ├── Latency benchmark (p50, p95, p99)                       │       │
│  │ ├── Memory profiling                                         │       │
│  │ ├── Gate: p99 < SLO threshold                                │       │
│  │ └── Report: benchmark_results.json                           │       │
│  └────────────────────────┬─────────────────────────────────────┘       │
│                           ▼                                             │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │ Stage 5: DEPLOY                                              │       │
│  │ ├── Build Docker image                                       │       │
│  │ ├── Deploy canary (10% traffic)                              │       │
│  │ ├── Gate: SLOs met for 30 minutes                            │       │
│  │ ├── Promote to production (100% traffic)                     │       │
│  │ └── Update model registry                                    │       │
│  └──────────────────────────────────────────────────────────────┘       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Validation Gates

Each gate is a boolean predicate that must pass before proceeding:

| Gate | Condition | Failure Action |
|:---|:---|:---|
| Training quality | $\text{acc}_{\text{val}} > \tau_{\text{min}}$ | Block export |
| Export parity | $\max|f_\theta(x) - G(x)| < 10^{-5}$ | Block optimization |
| Quantization accuracy | $\text{acc}_{\text{FP32}} - \text{acc}_{\text{INT8}} < 2\%$ | Fall back to FP32 |
| Latency SLO | $p_{99} < \tau_{\text{SLO}}$ | Block deployment |
| Canary health | Error rate < 0.1% for 30 min | Rollback |

### Example GitHub Actions Workflow

```yaml
name: ML Pipeline
on: [push]
jobs:
  train-export-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Train
        run: python train_model.py --epochs 20
      - name: Export & Validate
        run: python export_and_optimize.py --checkpoint last.pt
      - name: Benchmark
        run: python benchmark.py --model model.onnx
      - name: Deploy
        if: github.ref == 'refs/heads/main'
        run: ./deploy.sh canary
```

In [ ]:
# Simulate the full CI/CD pipeline with validation gates

class PipelineGate:
    """A validation gate that blocks pipeline progression on failure."""

    def __init__(self, name: str, condition_fn, failure_msg: str):
        self.name = name
        self.condition_fn = condition_fn
        self.failure_msg = failure_msg

    def check(self, **kwargs) -> bool:
        passed = self.condition_fn(**kwargs)
        status = 'PASS' if passed else 'FAIL'
        symbol = '✓' if passed else '✗'
        print(f"  [{symbol}] {self.name}: {status}")
        if not passed:
            print(f"      → {self.failure_msg}")
        return passed


def run_pipeline():
    """Execute the full ML CI/CD pipeline with gates."""
    print("═" * 60)
    print("  ML CI/CD PIPELINE EXECUTION")
    print("═" * 60)

    # Stage 1: Training
    print("\n▶ Stage 1: TRAIN")
    train_acc = history['accuracy'][-1]
    train_loss = history['loss'][-1]
    print(f"  Training complete: loss={train_loss:.4f}, acc={train_acc:.3f}")

    gate_train = PipelineGate(
        'Training Quality',
        lambda acc, **kw: acc > 0.05,
        'Accuracy below minimum threshold'
    )
    if not gate_train.check(acc=train_acc):
        return False

    # Stage 2: Export + Parity
    print("\n▶ Stage 2: EXPORT + VALIDATE")
    max_parity_err = max_abs.max()
    print(f"  Exported to ONNX. Max parity error: {max_parity_err:.2e}")

    gate_parity = PipelineGate(
        'Export Parity',
        lambda err, **kw: err < 1e-5,
        'Parity test failed — export may be incorrect'
    )
    if not gate_parity.check(err=max_parity_err):
        return False

    # Stage 3: Quantization
    print("\n▶ Stage 3: OPTIMIZE + QUANTIZE")
    pred_agreement = quant_match.mean()
    print(f"  INT8 prediction agreement: {pred_agreement*100:.1f}%")

    gate_quant = PipelineGate(
        'Quantization Accuracy',
        lambda agreement, **kw: agreement > 0.90,
        'Too many prediction disagreements after quantization'
    )
    if not gate_quant.check(agreement=pred_agreement):
        return False

    # Stage 4: Benchmark
    print("\n▶ Stage 4: BENCHMARK")
    p99_lat = bench_results['int8'][1]['p99']
    slo_threshold = 100.0
    print(f"  INT8 p99 latency (batch=1): {p99_lat:.3f} ms (SLO: <{slo_threshold} ms)")

    gate_latency = PipelineGate(
        'Latency SLO',
        lambda lat, threshold, **kw: lat < threshold,
        'p99 latency exceeds SLO'
    )
    if not gate_latency.check(lat=p99_lat, threshold=slo_threshold):
        return False

    # Stage 5: Deploy
    print("\n▶ Stage 5: DEPLOY")
    print("  Deploying canary (simulated)...")
    print(f"  Model hash: {model_hash}")
    print(f"  FP32 path:  {onnx_fp32_path}")
    print(f"  INT8 path:  {onnx_int8_path}")

    print("\n" + "═" * 60)
    print("  PIPELINE RESULT: ALL GATES PASSED")
    print("═" * 60)
    return True


pipeline_success = run_pipeline()

<a id='section-12'></a>
## Section 12: Full Working Example — Complete Pipeline

This section demonstrates the entire pipeline end-to-end with a single
cohesive example, bringing together all previous sections.

### Pipeline Formal Specification

$$\Pi_{\text{CIFAR}} = (\mathcal{T}_{\text{SGD}}, \mathcal{E}_{\text{ONNX17}}, \mathcal{O}_{\text{ORT\_ALL}}, \mathcal{Q}_{\text{INT8}}, \mathcal{D}_{\text{FastAPI}}, \mathcal{M}_{\text{PSI}})$$

With invariants:

$$\|f_\theta(x) - G_{\text{FP32}}(x)\|_\infty < 10^{-5} \quad \text{(export parity)}$$

$$\|G_{\text{FP32}}(x) - G_{\text{INT8}}(x)\|_\infty < 1.0 \quad \text{(quantization tolerance)}$$

$$\text{argmax}\, G_{\text{FP32}}(x) = \text{argmax}\, G_{\text{INT8}}(x) \quad \text{(>90% of samples)}$$

In [ ]:
# Full end-to-end pipeline demonstration

print("╔════════════════════════════════════════════════════════════╗")
print("║     COMPLETE END-TO-END PIPELINE DEMONSTRATION            ║")
print("╚════════════════════════════════════════════════════════════╝")

# --- Step 1: Model setup ---
print("\n[1/6] MODEL SETUP")
torch.manual_seed(0)
pipeline_model = SmallCnn()

# Quick training with synthetic data
pipeline_model.train()
opt = optim.Adam(pipeline_model.parameters(), lr=1e-3)
for step in range(50):
    x_batch = torch.randn(32, 3, 32, 32)
    y_batch = torch.randint(0, 10, (32,))
    loss = F.cross_entropy(pipeline_model(x_batch), y_batch)
    opt.zero_grad()
    loss.backward()
    opt.step()
pipeline_model.eval()
print(f"  Model trained ({sum(p.numel() for p in pipeline_model.parameters()):,} params)")
print(f"  Final training loss: {loss.item():.4f}")

# --- Step 2: ONNX Export ---
print("\n[2/6] ONNX EXPORT")
pipeline_onnx_path = '/tmp/pipeline_model.onnx'
dummy = torch.randn(1, 3, 32, 32)
torch.onnx.export(
    pipeline_model, dummy, pipeline_onnx_path,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
onnx.checker.check_model(onnx.load(pipeline_onnx_path))
print(f"  Exported to {pipeline_onnx_path}")
print(f"  File size: {Path(pipeline_onnx_path).stat().st_size / 1024:.1f} KB")

# --- Step 3: Parity verification ---
print("\n[3/6] PARITY VERIFICATION")
pipe_sess_fp32 = ort.InferenceSession(pipeline_onnx_path,
                                       providers=['CPUExecutionProvider'])
max_errors = []
for _ in range(200):
    x_t = torch.randn(1, 3, 32, 32)
    with torch.no_grad():
        y_pt = pipeline_model(x_t).numpy()
    y_ort = pipe_sess_fp32.run(None, {'input': x_t.numpy()})[0]
    max_errors.append(np.max(np.abs(y_pt - y_ort)))

max_err = max(max_errors)
parity_pass = max_err < 1e-5
print(f"  Max parity error: {max_err:.2e}")
print(f"  Parity test: {'PASS' if parity_pass else 'FAIL'}")

# --- Step 4: Quantization ---
print("\n[4/6] INT8 QUANTIZATION")
pipeline_int8_path = '/tmp/pipeline_model_int8.onnx'
quantize_dynamic(
    model_input=pipeline_onnx_path,
    model_output=pipeline_int8_path,
    weight_type=QuantType.QUInt8,
)
fp32_sz = Path(pipeline_onnx_path).stat().st_size
int8_sz = Path(pipeline_int8_path).stat().st_size
print(f"  FP32: {fp32_sz/1024:.1f} KB → INT8: {int8_sz/1024:.1f} KB ({fp32_sz/int8_sz:.1f}× compression)")

pipe_sess_int8 = ort.InferenceSession(pipeline_int8_path,
                                       providers=['CPUExecutionProvider'])
match_count = 0
for _ in range(200):
    x_t = np.random.randn(1, 3, 32, 32).astype(np.float32)
    y_fp32 = pipe_sess_fp32.run(None, {'input': x_t})[0]
    y_int8 = pipe_sess_int8.run(None, {'input': x_t})[0]
    if y_fp32.argmax() == y_int8.argmax():
        match_count += 1
print(f"  Prediction agreement: {match_count}/200 ({match_count/200*100:.0f}%)")

# --- Step 5: Benchmark ---
print("\n[5/6] BENCHMARK")
for label, sess in [('FP32', pipe_sess_fp32), ('INT8', pipe_sess_int8)]:
    r = benchmark_session(sess, 'input', (1, 3, 32, 32), n_warmup=50, n_runs=500)
    print(f"  {label}: mean={r['mean']:.3f}ms  p50={r['p50']:.3f}ms  "
          f"p95={r['p95']:.3f}ms  p99={r['p99']:.3f}ms")

# --- Step 6: Deployment readiness ---
print("\n[6/6] DEPLOYMENT READINESS")
server_pipe = ONNXModelServer(pipeline_int8_path)
health = server_pipe.health()
print(f"  Health check: {health['status']}")
print(f"  Model hash:   {health['model_hash']}")
print(f"  Providers:    {health['providers']}")

test_result = server_pipe.predict(np.random.randn(1, 3, 32, 32).astype(np.float32))
print(f"  Test prediction: {test_result['predictions'][0]} "
      f"(confidence: {test_result['confidences'][0]:.3f})")

print("\n" + "═" * 60)
print("  PIPELINE COMPLETE — Model ready for production deployment")
print("═" * 60)

In [ ]:
# Final summary visualization: pipeline overview

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# 1. Training convergence
axes[0, 0].plot(history['loss'], 'o-', color='#e74c3c', linewidth=2, markersize=5)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('1. Training Convergence', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# 2. Parity error distribution
axes[0, 1].hist(max_abs, bins=30, color='#3498db', edgecolor='black', alpha=0.85)
axes[0, 1].axvline(1e-5, color='red', linestyle='--', linewidth=2, label='ε = 1e-5')
axes[0, 1].set_xlabel('Max |PT − ORT|')
axes[0, 1].set_title('2. Export Parity', fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(alpha=0.3)

# 3. Graph optimization
opt_names = list(optimization_results.keys())
opt_nodes = [optimization_results[n]['nodes'] for n in opt_names]
opt_colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
bars = axes[0, 2].bar(opt_names, opt_nodes, color=opt_colors, edgecolor='black')
axes[0, 2].set_ylabel('Node Count')
axes[0, 2].set_title('3. Graph Optimization', fontweight='bold')
axes[0, 2].grid(axis='y', alpha=0.3)
for bar, n in zip(bars, opt_nodes):
    axes[0, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    str(n), ha='center', fontweight='bold')

# 4. Quantization comparison
quant_labels = ['FP32', 'INT8']
quant_sizes = [fp32_size/1024, int8_size/1024]
bars = axes[1, 0].bar(quant_labels, quant_sizes, color=['#3498db', '#e67e22'],
                       edgecolor='black', width=0.5)
axes[1, 0].set_ylabel('Size (KB)')
axes[1, 0].set_title('4. Quantization Impact', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)
for bar, s in zip(bars, quant_sizes):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{s:.0f}KB', ha='center', fontweight='bold')

# 5. Benchmark comparison (batch=1)
bench_labels = ['FP32\nmean', 'FP32\np95', 'INT8\nmean', 'INT8\np95']
bench_vals = [
    bench_results['fp32'][1]['mean'], bench_results['fp32'][1]['p95'],
    bench_results['int8'][1]['mean'], bench_results['int8'][1]['p95'],
]
bench_colors = ['#3498db', '#2980b9', '#e67e22', '#d35400']
bars = axes[1, 1].bar(bench_labels, bench_vals, color=bench_colors, edgecolor='black')
axes[1, 1].set_ylabel('Latency (ms)')
axes[1, 1].set_title('5. Benchmark Results', fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)
for bar, v in zip(bars, bench_vals):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

# 6. Pipeline status
stages = ['Train', 'Export', 'Optimize', 'Quantize', 'Benchmark', 'Deploy']
stage_status = [1, 1, 1, 1, 1, 1]
stage_colors = ['#2ecc71' if s else '#e74c3c' for s in stage_status]
axes[1, 2].barh(stages, stage_status, color=stage_colors, edgecolor='black')
axes[1, 2].set_xlim(0, 1.5)
axes[1, 2].set_title('6. Pipeline Status', fontweight='bold')
for i, (stage, status) in enumerate(zip(stages, stage_status)):
    label = 'PASS' if status else 'FAIL'
    axes[1, 2].text(1.1, i, label, va='center', fontweight='bold',
                    fontsize=11, color='#2ecc71' if status else '#e74c3c')
axes[1, 2].set_xticks([])

plt.suptitle('End-to-End ML Pipeline with ONNX — Summary Dashboard',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

<a id='section-13'></a>
## Summary

### Pipeline Stages and Their Formal Guarantees

| Stage | Transformation | Correctness Guarantee |
|:---|:---|:---|
| **Train** | $\mathcal{D} \to \theta^*$ | $\mathcal{L}(\theta^*) \leq \mathcal{L}(\theta_0)$ (loss decreases) |
| **Export** | $f_\theta \to G_{\text{ONNX}}$ | $\|f_\theta(x) - G(x)\|_\infty < 10^{-5}$ |
| **Optimize** | $G \to G'$ | $\|G(x) - G'(x)\|_\infty < 10^{-5}$ (semantic preservation) |
| **Quantize** | $G' \to G_q$ | $\text{argmax}\,G'(x) = \text{argmax}\,G_q(x)$ for >90% of $x$ |
| **Deploy** | $G_q \to \text{API}$ | Correct HTTP contract, health endpoint, versioned |
| **Monitor** | $\text{API} \to \text{Metrics}$ | PSI < 0.2, p99 < SLO, error rate < 0.1% |

### Critical Formulas

**Cross-Entropy Loss:**
$$\mathcal{L}(\hat{y}, y) = -\sum_k y_k \log \text{softmax}(\hat{y})_k$$

**Quantization:**
$$Q(r) = \text{round}(r/s) + z, \quad \hat{r} = s \cdot (Q(r) - z), \quad |r - \hat{r}| \leq s/2$$

**Population Stability Index:**
$$\text{PSI} = \sum_i (p_i - q_i) \ln(p_i / q_i)$$

**Fusion Memory Savings:**
$$\Delta T = (k-1) \times 2 \times |T| \times \text{sizeof}$$

### Interview Takeaways

1. **Pipeline is a formal composition**: Each stage has defined inputs, outputs, and invariants
2. **Parity testing is non-negotiable**: Always verify export with multiple random inputs
3. **Quantization is a tradeoff**: Measure accuracy degradation explicitly before deploying
4. **Benchmarking requires methodology**: Warmup, percentiles, statistical significance
5. **Monitoring is continuous**: Drift detection (PSI), SLO tracking, alerting
6. **Version everything**: Model hash, training config, data snapshot
7. **CI/CD gates prevent regressions**: Automated checks at each pipeline stage
8. **Rollback strategy is essential**: Always have a known-good version ready

---

**Next:** [End-to-End Project — Apply](./End_to_End_Project_Apply.ipynb)